# Huynen decomposition on geocoded data

Check the visual appearance and NaN behavior of the Huynen target matrix in SAR and geographic coordinates. Numerical comparison with C uses the separate SLC comparison notebook.

The geocoded input is a multilooked T3 matrix. These images are a visual check, not a pixel-by-pixel comparison with the SLC result.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
from dask.diagnostics import ProgressBar

from polsarpro.decompositions import huynen
from polsarpro.io import open_netcdf_beam

# Change to your data paths.
input_alos_slc = Path("/data/psp/test_files/SAN_FRANCISCO_ALOS1_slc.nc")
input_alos_geo = Path("/data/psp/test_files/SAN_FRANCISCO_ALOS1_geocoded_T3_7_look_az.nc")
output_basis = "pauli"  # "pauli" (T3) or "lexicographic" (C3)

In [ ]:
slc = open_netcdf_beam(input_alos_slc)
geo = open_netcdf_beam(input_alos_geo)

## Apply the decomposition

Use a 7 × 7 averaging window for the SLC input. The geocoded input has already been multilooked in azimuth, so it receives a smaller 3 × 3 window. Persist the results for repeated plotting.

In [ ]:
with ProgressBar():
    res_slc = huynen(slc, boxcar_size=(7, 7), output_basis=output_basis).persist()
with ProgressBar():
    res_geo = huynen(geo, boxcar_size=(3, 3), output_basis=output_basis).persist()

## Display target powers

Show the three diagonal elements of the target matrix in linear power units. The SLC display is subsampled in azimuth for readability. The panels use the same color limits but represent different grids and averaging windows. Inspect for unexpected artifacts, especially near masked regions.

In [ ]:
for var in ("m11", "m22", "m33"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    fig.suptitle(f"Huynen target {var} ({res_geo.attrs['poltype']})")
    for ax, values, title in zip(
        axes, (res_slc[var][::8], res_geo[var]), ("SLC", "Geocoded")
    ):
        values.plot.imshow(ax=ax, interpolation="none", vmin=0, vmax=1)
        ax.set_title(title)
    plt.tight_layout()
    plt.show()

## Inspect NaN behavior

Compare the geocoded input and target NaN masks. The shared boxcar reapplies the input NaN mask after averaging, and Huynen masks all target elements wherever an averaged input element is NaN.

In [ ]:
with ProgressBar():
    input_nan = geo.to_array().isnull().any("variable").compute()
    output_nan = res_geo.to_array().isnull().any("variable").compute()

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, mask, title in zip(
    axes, (input_nan, output_nan), ("Geocoded input NaNs", "Huynen target NaNs")
):
    mask.plot.imshow(ax=ax, interpolation="none", vmin=0, vmax=1, cmap="gray_r")
    ax.set_title(title)
plt.tight_layout()
plt.show()